In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import warnings
import os
import json

warnings.filterwarnings('ignore')

print("✅ Libraries imported and warnings suppressed")


✅ Libraries imported and warnings suppressed


In [2]:
DATA_PATH = "C:/Users/sneha/Desktop/ecopackai/Dataset/integrated_dataset.csv"

try:
    df = pd.read_csv(DATA_PATH)
    print(f"\n✓ Loaded dataset: {df.shape}")
except FileNotFoundError:
    print(f"\n❌ Error: File not found at {DATA_PATH}")
    exit(1)

print(f"Initial columns: {len(df.columns)}")



✓ Loaded dataset: (606000, 24)
Initial columns: 24


In [3]:
def calculate_co2_impact_index(df):
    scaler = MinMaxScaler()
    co2_norm = scaler.fit_transform(df[['co2_emission_per_kg']])
    carbon_norm = scaler.fit_transform(df[['carbon_footprint']])
    bio_norm = scaler.fit_transform(df[['biodegradation_days']])
    recycle_norm = 1 - (df['recyclability_percent'] / 100)
    
    co2_index_raw = (
        0.40 * co2_norm.flatten() +
        0.30 * carbon_norm.flatten() +
        0.20 * bio_norm.flatten() +
        0.10 * recycle_norm
    )
    
    return co2_index_raw * 100

df['co2_impact_index'] = calculate_co2_impact_index(df)

print("✓ CO₂ Impact Index created")
print(f"Range: {df['co2_impact_index'].min():.2f} - {df['co2_impact_index'].max():.2f}")
print(f"Mean: {df['co2_impact_index'].mean():.2f} (Lower = Better Environmentally)")


✓ CO₂ Impact Index created
Range: 0.50 - 91.30
Mean: 23.85 (Lower = Better Environmentally)


In [4]:
def calculate_cost_efficiency_index(df):
    scaler = MinMaxScaler()
    cost_norm = scaler.fit_transform(df[['cost_per_unit_usd']])
    cost_score = 1 - cost_norm.flatten()
    
    reuse_score = df['reusability_percent'] / 100
    weight_efficiency = cost_score
    
    cei_raw = (
        0.50 * cost_score +
        0.30 * reuse_score +
        0.20 * weight_efficiency
    )
    
    return cei_raw * 100

df['cost_efficiency_index'] = calculate_cost_efficiency_index(df)

print("✓ Cost Efficiency Index created")
print(f"Range: {df['cost_efficiency_index'].min():.2f} - {df['cost_efficiency_index'].max():.2f}")
print(f"Mean: {df['cost_efficiency_index'].mean():.2f} (Higher = More cost-efficient)")


✓ Cost Efficiency Index created
Range: 29.18 - 79.46
Mean: 66.30 (Higher = More cost-efficient)


In [5]:
def calculate_material_suitability_score(df):
    scores = []
    for _, row in df.iterrows():
        score = 0
        
        # Category Match (40 points)
        if pd.notna(row['suitable_categories']):
            categories = str(row['suitable_categories']).lower()
            product_cat = str(row['product_category']).lower()
            if product_cat in categories:
                score += 40
            elif any(word in categories for word in product_cat.split()):
                score += 20
        
        # Load Handling vs Fragility (30 points)
        fragility = row['fragility_index']
        load_score = row['load_handling_score']
        if fragility >= 4:
            if load_score >= 7: score += 30
            elif load_score >= 5: score += 20
            else: score += 5
        elif fragility >= 2:
            if load_score >= 5: score += 30
            elif load_score >= 3: score += 20
            else: score += 10
        else:
            if load_score >= 3: score += 30
            else: score += 20
        
        # Physical Requirements (20 points)
        moisture = row['moisture_resistance']
        thermal = row['thermal_resistance']
        if row['product_category'] in ['Food', 'Cosmetics']:
            if moisture >= 7: score += 10
            elif moisture >= 5: score += 5
        else:
            if moisture >= 5: score += 10
        if row['product_category'] == 'Electronics':
            if thermal >= 7: score += 10
            elif thermal >= 5: score += 5
        else:
            if thermal >= 5: score += 10
        
        # General Quality (10 points)
        avg_resistance = (moisture + thermal + load_score) / 3
        if avg_resistance >= 7: score += 10
        elif avg_resistance >= 5: score += 5
        
        scores.append(min(score, 100))
    return scores

df['material_suitability_score'] = calculate_material_suitability_score(df)

print("✓ Material Suitability Score created")
print(f"Range: {df['material_suitability_score'].min():.2f} - {df['material_suitability_score'].max():.2f}")
print(f"Mean: {df['material_suitability_score'].mean():.2f} (Higher = Better Match)")


✓ Material Suitability Score created
Range: 5.00 - 100.00
Mean: 50.21 (Higher = Better Match)


In [6]:
output_path = "C:/Users/sneha/Desktop/ecopackai/Dataset/engineered_dataset.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

df.to_csv(output_path, index=False)
print(f"✓ Engineered dataset saved: {output_path}")
print(f"Total columns: {len(df.columns)}")
print("New features added: 4")
print("  - co2_impact_index")
print("  - cost_efficiency_index")
print("  - material_suitability_score")
print("  - overall_sustainability_score")


✓ Engineered dataset saved: C:/Users/sneha/Desktop/ecopackai/Dataset/engineered_dataset.csv
Total columns: 27
New features added: 4
  - co2_impact_index
  - cost_efficiency_index
  - material_suitability_score
  - overall_sustainability_score


In [8]:
print(df.columns.tolist())


['product_id', 'product_name', 'product_category', 'product_weight_kg', 'fragility_index', 'shipping_type', 'material_id', 'material_type', 'packaging_type', 'suitable_categories', 'recyclability_percent', 'recyclability_category', 'biodegradation_days', 'carbon_footprint', 'co2_emission_per_kg', 'load_handling_score', 'moisture_resistance', 'thermal_resistance', 'cost_per_unit_usd', 'supplier_region', 'reusability_percent', 'recycled_content_percent', 'waste_reduction_impact', 'compatibility_score', 'co2_impact_index', 'cost_efficiency_index', 'material_suitability_score']


In [10]:
# Invert CO2 impact index so higher = better
co2_score_inverted = 100 - df['co2_impact_index']

# Calculate overall sustainability score
df['overall_sustainability_score'] = (
    0.40 * co2_score_inverted +
    0.30 * df['cost_efficiency_index'] +
    0.30 * df['material_suitability_score']
)

print("✓ Overall Sustainability Score created")
print(f"Range: {df['overall_sustainability_score'].min():.2f} - {df['overall_sustainability_score'].max():.2f}")
print(f"Mean: {df['overall_sustainability_score'].mean():.2f}")


✓ Overall Sustainability Score created
Range: 31.26 - 87.67
Mean: 65.41


In [11]:
engineered_features = [
    'co2_impact_index',
    'cost_efficiency_index', 
    'material_suitability_score',
    'overall_sustainability_score'
]

print("Engineered Features Statistics:")
df[engineered_features].describe().round(2)

# Feature metadata
feature_metadata = {
    "co2_impact_index": {
        "description": "Environmental impact score (0-100, lower is better)",
        "components": ["CO2 emission per kg (40%)", "Carbon footprint (30%)", 
                      "Biodegradation time (20%)", "Recyclability (10%)"],
        "range": [0, 100],
        "interpretation": "Lower values indicate more environmentally friendly materials"
    },
    "cost_efficiency_index": {
        "description": "Economic efficiency score (0-100, higher is better)",
        "components": ["Cost per unit (50%)", "Reusability (30%)", "Weight efficiency (20%)"],
        "range": [0, 100],
        "interpretation": "Higher values indicate more cost-effective solutions"
    },
    "material_suitability_score": {
        "description": "Product-material match score (0-100, higher is better)",
        "components": ["Category compatibility (40%)", "Load handling (30%)", 
                      "Physical requirements (20%)", "General quality (10%)"],
        "range": [0, 100],
        "interpretation": "Higher values indicate better material-product compatibility"
    },
    "overall_sustainability_score": {
        "description": "Combined sustainability score (0-100, higher is better)",
        "components": ["Environmental (40%)", "Economic (30%)", "Suitability (30%)"],
        "range": [0, 100],
        "interpretation": "Higher values indicate better overall packaging solution"
    }
}

metadata_path = "C:/Users/sneha/Desktop/ecopackai/Data/docs/feature_metadata.json"
os.makedirs(os.path.dirname(metadata_path), exist_ok=True)

with open(metadata_path, 'w') as f:
    json.dump(feature_metadata, f, indent=2)

print(f"✓ Feature metadata saved: {metadata_path}")


Engineered Features Statistics:
✓ Feature metadata saved: C:/Users/sneha/Desktop/ecopackai/Data/docs/feature_metadata.json
